In [1]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import cv2
import numpy as np

In [5]:
dataset_paths_real = []

folder_real = r"E:\dataset\Deepfake Detection Dataset\Preprocessed_Dataset\real"

for real in os.listdir(folder_real):
    path_real = os.path.join(folder_real,real)
    dataset_paths_real.append(path_real)

#display(dataset_paths_real)

In [7]:
dataset_paths_fake = []

folder_fake = r"E:\dataset\Deepfake Detection Dataset\Preprocessed_Dataset\fake"

for fake in os.listdir(folder_fake):
    path_fake = os.path.join(folder_fake,fake)
    dataset_paths_fake.append(path_fake)

#display(dataset_paths_fake)

In [9]:
#display(dataset_paths_real,dataset_paths_fake)

In [11]:
print("Real videos:", len(dataset_paths_real))
print("Fake videos:", len(dataset_paths_fake))
print("Total videos:", len(dataset_paths_real) + len(dataset_paths_fake))

Real videos: 1000
Fake videos: 1000
Total videos: 2000


In [13]:
import os
import cv2
from tqdm import tqdm

# ── Paths ────────────────────────────────────────────────────
folder_real = r"E:\dataset\Deepfake Detection Dataset\Preprocessed_Dataset\real"
folder_fake = r"E:\dataset\Deepfake Detection Dataset\Preprocessed_Dataset\fake"
output_base = r"E:\dataset\Deepfake Detection Dataset\frames_dataset"

# ── Config ───────────────────────────────────────────────────
FRAMES_TO_EXTRACT = 20

# ── Already have these from your existing code ───────────────
dataset_paths_real = []
for real in os.listdir(folder_real):
    dataset_paths_real.append(os.path.join(folder_real, real))

dataset_paths_fake = []
for fake in os.listdir(folder_fake):
    dataset_paths_fake.append(os.path.join(folder_fake, fake))


def extract_frames(video_path, output_folder, n_frames=20):
    """Extract n evenly-spaced frames from a video and save as JPEGs."""
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"  [SKIP] Could not open: {video_path}")
        return False
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < n_frames:
        print(f"  [WARN] Only {total_frames} frames in {os.path.basename(video_path)}, extracting all.")
        indices = list(range(total_frames))
    else:
        # Evenly spaced indices across full video duration
        step = total_frames / n_frames
        indices = [int(step * i + step / 2) for i in range(n_frames)]
    
    os.makedirs(output_folder, exist_ok=True)
    
    saved = 0
    for frame_num in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        if ret:
            frame_path = os.path.join(output_folder, f"frame_{saved+1:02d}.jpg")
            cv2.imwrite(frame_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
    
    cap.release()
    return True


def process_split(video_paths, label, n_frames=20):
    """Process all videos for a given label (real or fake)."""
    print(f"\nProcessing {label.upper()} videos ({len(video_paths)} total)...")
    
    success, skipped = 0, 0
    
    for video_path in tqdm(video_paths, desc=label):
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_folder = os.path.join(output_base, label, video_name)
        
        # Skip if already extracted (resume support)
        if os.path.exists(output_folder):
            existing = len(os.listdir(output_folder))
            if existing == n_frames:
                skipped += 1
                continue
        
        result = extract_frames(video_path, output_folder, n_frames)
        if result:
            success += 1
    
    print(f"  Done — Extracted: {success} | Skipped (already exists): {skipped}")


# ── Run ──────────────────────────────────────────────────────
process_split(dataset_paths_real, "real", FRAMES_TO_EXTRACT)
process_split(dataset_paths_fake, "fake", FRAMES_TO_EXTRACT)

print("\nFrame extraction complete.")
print(f"Output saved to: {output_base}")


Processing REAL videos (1000 total)...


real: 100%|████████████████████████████████████████████████████████████████████████| 1000/1000 [22:00<00:00,  1.32s/it]


  Done — Extracted: 1000 | Skipped (already exists): 0

Processing FAKE videos (1000 total)...


fake: 100%|████████████████████████████████████████████████████████████████████████| 1000/1000 [22:20<00:00,  1.34s/it]

  Done — Extracted: 1000 | Skipped (already exists): 0

Frame extraction complete.
Output saved to: E:\dataset\Deepfake Detection Dataset\frames_dataset
